# 🔐 Fabric Notebook Secret Scanner

Scans **all notebooks across all accessible workspaces** for hardcoded secrets,
connection keys, passwords, SAS tokens, and API keys.

| What it does | How |
|---|---|
| Enumerates workspaces | Fabric REST API |
| Downloads notebook definitions | Base64-decoded cell content |
| Pattern matching | Regex for 20+ secret patterns |
| Output | Spark DataFrame + optional Delta table |

> **Permissions required:** Workspace Viewer (or higher) on target workspaces.  
> **Runtime:** Fabric Spark with PySpark kernel.

## ⚙️ Configuration

In [ ]:
# --- CONFIGURATION ---

# Set to True to save results to a Delta table in the attached lakehouse
SAVE_TO_LAKEHOUSE = False
DELTA_TABLE_NAME = "notebook_secret_scan_results"

# Workspaces to skip (by name substring) - e.g., system or template workspaces
SKIP_WORKSPACES = []

# Set to True to print each finding as it's discovered (verbose mode)
VERBOSE = True

## 🔑 Authentication & Helpers

In [ ]:
import requests
import base64
import json
import re
import time
from datetime import datetime

# Get token from the Fabric runtime
token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
BASE_URL = "https://api.fabric.microsoft.com/v1"


def fabric_get(path, params=None):
    """GET request against the Fabric REST API with automatic pagination."""
    url = f"{BASE_URL}{path}"
    all_items = []
    while url:
        resp = requests.get(url, headers=HEADERS, params=params)
        resp.raise_for_status()
        data = resp.json()
        all_items.extend(data.get("value", []))
        url = data.get("continuationUri")
        params = None  # continuation URI already includes params
    return all_items


def fabric_post(path, body=None):
    """POST request with long-running operation support."""
    url = f"{BASE_URL}{path}"
    resp = requests.post(url, headers=HEADERS, json=body)
    
    # Handle 202 Accepted (long-running operation)
    if resp.status_code == 202:
        location = resp.headers.get("Location")
        retry_after = int(resp.headers.get("Retry-After", 2))
        for _ in range(30):  # max ~60s wait
            time.sleep(retry_after)
            poll = requests.get(location, headers=HEADERS)
            if poll.status_code == 200:
                poll_data = poll.json()
                status = poll_data.get("status", "")
                if status == "Succeeded":
                    # Fetch the result from the definition endpoint
                    result_url = poll.headers.get("Location", location)
                    if result_url != location:
                        final = requests.get(result_url, headers=HEADERS)
                        final.raise_for_status()
                        return final.json()
                    return poll_data
                elif status == "Failed":
                    raise Exception(f"Operation failed: {poll_data}")
            elif poll.status_code == 200:
                return poll.json()
        raise TimeoutError("Long-running operation timed out")
    
    resp.raise_for_status()
    return resp.json()


print("✅ Authenticated successfully")

## 🔍 Secret Detection Patterns

In [ ]:
# Each pattern: (label, compiled regex)
# Patterns are case-insensitive and look for key=value or key:value assignments

SECRET_PATTERNS = [
    # Storage account keys
    ("Storage Account Key",
     re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),

    # Shared Access Signatures
    ("SAS Token",
     re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.IGNORECASE)),
    ("SAS Token (assigned)",
     re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']', re.IGNORECASE)),

    # SQL passwords
    ("SQL Password",
     re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.IGNORECASE)),

    # Cosmos DB keys
    ("Cosmos DB Key",
     re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),

    # Generic API keys
    ("API Key",
     re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}', re.IGNORECASE)),

    # Service principal / client secrets
    ("Client Secret",
     re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}', re.IGNORECASE)),

    # Connection strings (generic)
    ("Connection String",
     re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']', re.IGNORECASE)),

    # Bearer tokens hardcoded
    ("Hardcoded Bearer Token",
     re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.IGNORECASE)),

    # Azure Event Hub / Service Bus
    ("Event Hub / Service Bus Key",
     re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.IGNORECASE)),

    # OpenAI / Azure OpenAI keys
    ("OpenAI API Key",
     re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}', re.IGNORECASE)),
    ("Azure OpenAI Key",
     re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}', re.IGNORECASE)),

    # Generic secret assignment (catches many patterns)
    ("Generic Secret Assignment",
     re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']', re.IGNORECASE)),

    # Spark config with account keys
    ("Spark Config Account Key",
     re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.IGNORECASE)),

    # JDBC URLs with passwords
    ("JDBC Password",
     re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.IGNORECASE)),
]

# Lines to ignore (comments explaining secrets, imports, etc.)
IGNORE_PATTERNS = [
    re.compile(r'^\s*#'),       # Python comments
    re.compile(r'^\s*//'),      # JS-style comments
    re.compile(r'<your[_-]'),   # Placeholder prompts
    re.compile(r'\{\{'),        # Template variables
    re.compile(r'<replace'),    # Placeholder prompts
    re.compile(r'PLACEHOLDER', re.IGNORECASE),
    re.compile(r'example\.com', re.IGNORECASE),
]


def is_ignored(line):
    return any(p.search(line) for p in IGNORE_PATTERNS)


def scan_content(text):
    """Scan text and return list of (secret_type, line_number, snippet)."""
    findings = []
    for line_num, line in enumerate(text.split('\n'), start=1):
        if is_ignored(line):
            continue
        for label, pattern in SECRET_PATTERNS:
            if pattern.search(line):
                snippet = line.strip()[:120]
                findings.append((label, line_num, snippet))
                break  # one match per line is enough
    return findings


print(f"✅ Loaded {len(SECRET_PATTERNS)} secret detection patterns")

## 🚀 Scan All Workspaces

In [ ]:
# --- 1. List all workspaces ---
print("📂 Fetching workspaces...")
workspaces = fabric_get("/workspaces")
print(f"   Found {len(workspaces)} workspaces")

# Filter out skipped workspaces
if SKIP_WORKSPACES:
    workspaces = [ws for ws in workspaces
                  if not any(skip.lower() in ws['displayName'].lower() for skip in SKIP_WORKSPACES)]
    print(f"   Scanning {len(workspaces)} workspaces after filters")

# --- 2. Scan each workspace ---
all_findings = []   # (workspace, notebook, secret_type, line, snippet)
errors = []         # (workspace, notebook, error_msg)
scan_stats = {"workspaces": 0, "notebooks": 0, "findings": 0}

for ws in workspaces:
    ws_name = ws["displayName"]
    ws_id = ws["id"]
    scan_stats["workspaces"] += 1
    print(f"\n🔍 [{scan_stats['workspaces']}/{len(workspaces)}] {ws_name}")

    # List notebooks in this workspace
    try:
        notebooks = fabric_get(f"/workspaces/{ws_id}/items?type=Notebook")
    except Exception as e:
        errors.append((ws_name, "(list notebooks)", str(e)))
        print(f"   ⚠️  Could not list notebooks: {e}")
        continue

    if not notebooks:
        print(f"   No notebooks found")
        continue

    print(f"   Found {len(notebooks)} notebooks")

    for nb in notebooks:
        nb_name = nb["displayName"]
        nb_id = nb["id"]
        scan_stats["notebooks"] += 1

        try:
            # Get notebook definition
            defn = fabric_post(f"/workspaces/{ws_id}/notebooks/{nb_id}/getDefinition")

            # Extract and decode all parts
            parts = defn.get("definition", {}).get("parts", [])
            full_content = ""
            for part in parts:
                payload = part.get("payload", "")
                if payload:
                    try:
                        decoded = base64.b64decode(payload).decode("utf-8", errors="replace")
                        full_content += decoded + "\n"
                    except Exception:
                        full_content += payload + "\n"

            # Scan the content
            findings = scan_content(full_content)
            for secret_type, line_num, snippet in findings:
                all_findings.append((ws_name, nb_name, secret_type, line_num, snippet))
                scan_stats["findings"] += 1
                if VERBOSE:
                    print(f"   🚨 {nb_name} | Line {line_num} | {secret_type}")

        except Exception as e:
            errors.append((ws_name, nb_name, str(e)))
            if VERBOSE:
                print(f"   ⚠️  {nb_name}: {str(e)[:80]}")

print(f"\n{'='*60}")
print(f"✅ Scan complete!")
print(f"   Workspaces scanned: {scan_stats['workspaces']}")
print(f"   Notebooks scanned:  {scan_stats['notebooks']}")
print(f"   Secrets found:      {scan_stats['findings']}")
print(f"   Errors:             {len(errors)}")

## 📊 Results

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# --- Build findings DataFrame ---
if all_findings:
    schema = StructType([
        StructField("Workspace", StringType(), True),
        StructField("Notebook", StringType(), True),
        StructField("Secret_Type", StringType(), True),
        StructField("Line_Number", IntegerType(), True),
        StructField("Code_Snippet", StringType(), True),
    ])
    df_findings = spark.createDataFrame(
        [Row(Workspace=f[0], Notebook=f[1], Secret_Type=f[2],
             Line_Number=f[3], Code_Snippet=f[4]) for f in all_findings],
        schema=schema
    )
    print(f"🚨 Found {df_findings.count()} hardcoded secrets:\n")
    df_findings.show(100, truncate=False)
else:
    print("✅ No hardcoded secrets found — all clean!")
    df_findings = None

# --- Build errors DataFrame ---
if errors:
    df_errors = spark.createDataFrame(
        [Row(Workspace=e[0], Notebook=e[1], Error=e[2]) for e in errors]
    )
    print(f"\n⚠️  {len(errors)} errors encountered:\n")
    df_errors.show(50, truncate=False)

## 📈 Summary by Secret Type

In [ ]:
if df_findings:
    print("Findings by secret type:\n")
    df_findings.groupBy("Secret_Type").count().orderBy("count", ascending=False).show(truncate=False)

    print("\nFindings by workspace:\n")
    df_findings.groupBy("Workspace").count().orderBy("count", ascending=False).show(truncate=False)

    print("\nTop offending notebooks:\n")
    df_findings.groupBy("Workspace", "Notebook").count().orderBy("count", ascending=False).show(20, truncate=False)

## 💾 Save to Lakehouse (Optional)

In [ ]:
if SAVE_TO_LAKEHOUSE and df_findings:
    from pyspark.sql.functions import current_timestamp

    df_out = df_findings.withColumn("scanned_at", current_timestamp())
    df_out.write.mode("append").format("delta").saveAsTable(DELTA_TABLE_NAME)
    print(f"💾 Saved {df_out.count()} findings to lakehouse table '{DELTA_TABLE_NAME}'")
elif SAVE_TO_LAKEHOUSE:
    print("ℹ️  No findings to save.")
else:
    print("ℹ️  Lakehouse save disabled. Set SAVE_TO_LAKEHOUSE = True to persist results.")